# Data Analysis 

LSE ID = 

## Loading the tables from the database

First, I start importing all the relevant libraries.

In [12]:
import pandas as pd
import plotly.express as px
import sqlite3

In [13]:
conn = sqlite3.connect("../data/movies.db")

Then, I select the data from the tables in the database

In [14]:
movies = pd.read_sql("SELECT * FROM movies", conn)
genres = pd.read_sql("SELECT * FROM genres", conn)
genres_movie = pd.read_sql("SELECT * FROM movie_genres", conn)



## Distribution of runtime per year and per genre

First, I explored how did the distribution of runtime fluctuated per year. For that I used the movies table, and the relevant columns. Runtime is stable at ~100–130 min as the dominant peak across all years (2015–2025) — no clear trend of films getting longer or shorter over time. 2020–2021 show more dispersed/noisy distributions, with several outliers near 0 and a wider tail — likely a COVID effect (fewer films, different mix of releases/streaming).
2023–2025 show longer, more pronounced right tails (outliers up to 200–220 min), suggesting more extended-runtime films in recent years.
Movie volume looks relatively consistent year to year (~60–70 at the peak bin), aside from the dispersion noted in 2020.
Near-zero runtime values appear in several years (2018, 2020, 2022, 2024) — likely data errors or missing-runtime placeholders.

In [31]:
fig_years= px.histogram(movies, x="runtime" , facet_row="year", width=500, height=2000)
fig_years.update_yaxes(title_text="Number of movies")


The runtime seems to be consistent across years

In [16]:
q_genres = pd.read_sql("""

SELECT 
    m.year,
    m.id,
    m.title,
    g.genre_name,
    m.runtime
FROM movies m
JOIN movie_genres g ON m.id = g.id
ORDER BY m.year ASC

""", conn) 

In [17]:
q_genres

,year,id,title,genre_name,runtime
0,2015,140607,Star Wars: The Force Awakens,Adventure,136
1,2015,140607,Star Wars: The Force Awakens,Action,136
2,2015,140607,Star Wars: The Force Awakens,Science Fiction,136
3,2015,135397,Jurassic World,Adventure,124
4,2015,135397,Jurassic World,Science Fiction,124
...,...,...,...,...,...
3170,2025,1233575,Black Bag,Mystery,94
3171,2025,1233575,Black Bag,Thriller,94
3172,2025,701387,Bugonia,Science Fiction,119
3173,2025,701387,Bugonia,Thriller,119


In [18]:
fig_genres= px.histogram(q_genres, x="runtime" , facet_row="genre_name", labels={"genre_name": 'genre', "count":"Number of movies"}, height=5000, width=1000)
fig_genres.update_yaxes(title_text="Number of movies")

the dispersion varies between genres

## Analysis


In [19]:
q_genres

,year,id,title,genre_name,runtime
0,2015,140607,Star Wars: The Force Awakens,Adventure,136
1,2015,140607,Star Wars: The Force Awakens,Action,136
2,2015,140607,Star Wars: The Force Awakens,Science Fiction,136
3,2015,135397,Jurassic World,Adventure,124
4,2015,135397,Jurassic World,Science Fiction,124
...,...,...,...,...,...
3170,2025,1233575,Black Bag,Mystery,94
3171,2025,1233575,Black Bag,Thriller,94
3172,2025,701387,Bugonia,Science Fiction,119
3173,2025,701387,Bugonia,Thriller,119


In [20]:
order = (
    q_genres.groupby("genre_name")["runtime"]
    .apply(lambda s: s.quantile(0.75) - s.quantile(0.25))
    .sort_values().index.to_list()

)



In [22]:
px.box(q_genres, x="runtime", y="genre_name", color = "genre_name", points = "outliers", category_orders={"genre_name":order})

In [25]:
q_genres["genre_name"].value_counts()

genre_name
Action             451
Adventure          368
Comedy             363
Drama              341
Thriller           240
Fantasy            200
Science Fiction    190
Family             185
Animation          173
Crime              151
Horror             140
Romance            107
Mystery             98
History             77
War                 42
Music               37
Western              7
Documentary          5
Name: count, dtype: int64

q1_df

In [29]:
px.box(q_genres, x="year", y="runtime", color = "year", points = "outliers")

In [30]:
fig = px.line(movies.groupby("year")["runtime"].mean(), range_y=[0,130])
fig.show()

In [28]:
mov_2020 = q1_df[(q1_df["year"] == 2020) & (q1_df["runtime"]<45) ]

In [29]:
mov_2020

,year,id,title,genre_name,runtime
1473,2020,1198553,Nidja's Kitchen 2,Comedy,4
1474,2020,1198553,Nidja's Kitchen 2,Drama,4
1520,2020,1029460,The Western,Western,10
1712,2020,972402,Lost Generation,War,25
1713,2020,972402,Lost Generation,Drama,25
1714,2020,972402,Lost Generation,History,25


In [74]:
q1_df["year"].dtype

dtype('int64')